# Poisson sanity: boundary-value droplet initialization and finite-time stability

64×64 periodic Active Model B에서 원형 droplet이 유지되는지 확인한다.
`CASES`에서 g, ell, lambda, D, nominal radius, a/b/kappa를 직접 바꾼다.
Euler–Maruyama만 사용한다. 이 노트북은 morphology 실험이며 EP는 계산하지 않아 **D=0**도 정확히 지원한다.

**초기화**
1. local passive radial equation의 boundary-value problem(BVP)을 푼다.
2. 기본 `INITIALIZATION="periodic_stationary"`에서는 그 해를 실제 periodic lattice의
   Poisson+active stationary equation으로 continuation 보정한다.
3. reference와 같은 질량인 작은 m=2 shape perturbation을 만든다.
4. reference/perturbed pair를 같은 noise로 진화시킨다.

정상해가 존재한다고 안정한 것은 아니다. 수렴하지 않았다고 해가 없다고 결론내리지 않는다.
실패한 BVP/root를 tanh 초기장으로 대체하지 않으며, 초기화 실패는 stability 결과와 구분한다.
`INITIALIZATION="local_bvp"`를 선택하면 동일한 local BVP droplet에서 g/ell을 바꾸는 quench 실험이 된다.
이 경우 초기 relaxation은 실제 Poisson+active 정상해 주변의 안정성과 구별해야 한다.

플롯·영상·집계 코드는 셀에서 직접 편집한다. 생성 결과는 조건별로 cache한다.
노트북은 실행 결과 없이 제공된다.
BVP 구현 참고: [SciPy solve_bvp의 unknown parameters 및 singular term S](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_bvp.html).


In [ ]:
from pathlib import Path
from functools import lru_cache
import hashlib
import json
import math
import sys
import time
import numpy as np
from html import escape
import torch
from scipy.integrate import solve_bvp, cumulative_trapezoid
from scipy.optimize import root, brentq
from scipy.ndimage import map_coordinates
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import display, HTML
from tqdm.auto import tqdm

HERE = Path.cwd().resolve()
roots = [HERE, *HERE.parents, HERE / "KNEEP"]
KNEEP_ROOT = next((p for p in roots if (p / "shell_force.py").exists()), None)
if KNEEP_ROOT is None:
    raise FileNotFoundError("Set KNEEP_ROOT to your KNEEP repository.")
LRAMB_ROOT = KNEEP_ROOT.parent / "LRAMB"
sys.path.insert(0, str(LRAMB_ROOT))
from lramb.model import KernelSpec
from lramb.circle import cluster_observables

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(1)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True,
                     "grid.alpha": 0.2, "animation.embed_limit": 100})
def display_table(rows):
    # Only rendering; the underlying records remain ordinary editable dictionaries.
    rows = list(rows)
    if not rows:
        print("(empty table)")
        return
    columns = list(dict.fromkeys(key for row in rows for key in row))
    header = "<tr>"+"".join("<th>"+escape(str(key))+"</th>" for key in columns)+"</tr>"
    body = "".join("<tr>"+"".join("<td>"+escape(str(row.get(key,"")))+"</td>"
                                for key in columns)+"</tr>" for row in rows)
    display(HTML('<div style="overflow-x:auto"><table>'+header+body+"</table></div>"))

print("Device:", DEVICE)


## 1. Editable parameter cases

기본 target은 **g=1, ell=8, lambda=0.8, R_nominal=12**다.
local passive/active와 Poisson passive를 controls로 포함한다.
D=0에서 deterministic shape relaxation을 먼저 보고 D=0.1 case와 비교한다.
`CASES = CASES[:4]` 등으로 필요한 조건만 먼저 실행할 수 있다.

radius 변경은 총질량도 바꾼다. 같은 local bulk/radius의 g/ell/lambda sweep에서는 동일한 질량을 유지한다.
`REMOVE_ZERO_MODE=True`는 Khat(0)=0이다. False이면 Poisson의 Khat(0)=1을 보존한다.
고정질량 dynamics에서 둘의 차이는 chemical potential의 상수항이며, 기본적으로 같은 force를 준다.


In [ ]:
GRID = dict(n=32, dx=1.0)
BASE = dict(a=-1.0, b=1.0, kappa=1.0, mobility=1.0,
            g=0.3, ell=1.0, lam=0.8, D=0.0, radius=8.0)
CASES = [
    dict(name="local passive", g=0.0, lam=0.0),
    dict(name="local active", g=0.0),
    dict(name="Poisson passive", lam=0.0),
    dict(name="target"),
    dict(name="weaker g", g=0.1),
    dict(name="stronger g", g=0.5),
    dict(name="shorter ell=2", ell=0.5),
    dict(name="shorter ell=4", ell=2.0),
    dict(name="smaller droplet", radius=6.0),
    dict(name="larger droplet", radius=10.0),
    dict(name="target with noise", D=0.1),
    # dict(name="stiffer interface", kappa=2.0),
    # dict(name="more active", lam=1.2),
]
CASES = [{**BASE, **case} for case in CASES]
INITIALIZATION = "periodic_stationary"    # or "local_bvp" for a quench experiment
REMOVE_ZERO_MODE = True
BVP = dict(nodes=400, tol=1e-5, max_nodes=12000)
ROOT = dict(tol=1e-7, maxiter=100, continuation_steps=8)
RUN = dict(dt=0.01, duration=160.0, save_every=1.0, seed=17)
PERTURB = dict(relative_amplitude=0.02, mode=2)
RULES = dict(radius_tolerance=0.10, circularity_min=0.95,
             largest_fraction_min=0.98, shape_linear_limit=0.15,
             fit_until=40.0, mass_tolerance=1e-10)
OUTPUT_ROOT = KNEEP_ROOT / "results" / "poisson_sanity"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
NUMERICS_VERSION = "radial-bvp-periodic-continuation-em-v1"  # change after editing numerical equations

n, dx = GRID["n"], GRID["dx"]
length, dV = n*dx, dx**2
outer_radius = length/2
if n < 16 or n % 2 or dx <= 0:
    raise ValueError("Use a positive dx and even n >= 16.")
if INITIALIZATION not in ("periodic_stationary", "local_bvp"):
    raise ValueError("Choose periodic_stationary or local_bvp.")
if not CASES:
    raise ValueError("Provide at least one case.")
if len({c["name"] for c in CASES}) != len(CASES):
    raise ValueError("Case names must be unique.")
for c in CASES:
    if (not np.isfinite(list(v for k,v in c.items() if k != "name")).all()
        or c["a"] >= 0 or min(c["b"], c["kappa"], c["ell"], c["mobility"]) <= 0
        or c["g"] < 0 or c["D"] < 0 or not 3*dx < c["radius"] < length/3):
        raise ValueError(f"Invalid local double well, kernel, noise, or radius: {c}")
if not 0 < PERTURB["relative_amplitude"] < 0.1 or not 2 <= PERTURB["mode"] < 64 or int(PERTURB["mode"]) != PERTURB["mode"]:
    raise ValueError("Use a small shape perturbation with integer mode >=2.")
if min(RUN["dt"], RUN["duration"], RUN["save_every"]) <= 0:
    raise ValueError("Time intervals must be positive.")
steps = round(RUN["duration"]/RUN["dt"])
stride = round(RUN["save_every"]/RUN["dt"])
if stride < 1 or not np.isclose(steps*RUN["dt"], RUN["duration"]) or not np.isclose(stride*RUN["dt"], RUN["save_every"]):
    raise ValueError("duration/save_every must be integer multiples of dt.")
display_table(CASES)
print("Frame count per pair:", 1 + steps//stride + int(steps % stride != 0))
print("Output:", OUTPUT_ROOT)


## 2. Boundary-value problem for a curved local droplet

$$
a\phi+b\phi^3-\kappa(\phi''+\phi'/r)=\mu_*,
\qquad \phi'(0)=\phi'(R_b)=0,
\qquad \frac{2}{R_b^2}\int_0^{R_b}r\phi(r)\,dr=\bar\phi_{\rm disk}.
$$

화학퍼텐셜 `mu_*`도 unknown이다. `bar_phi_disk`는 nominal radius의 sharp-interface 면적비로 정한다.
곡률 때문에 실제 plateau와 interface radius는 ±sqrt(-a/b), nominal radius와 달라질 수 있다.
**총질량과 두 Neumann 조건을 주며, 양쪽 plateau ±1까지 추가로 강제하지 않는다.**

원점의 -phi'/r singular term은 solve_bvp의 S 행렬로 처리한다.
tanh는 Newton 초기 추측에만 쓰며, 시뮬레이션 초기장은 수렴한 BVP 해에서 만든다.
disk 밖의 square corners는 외곽 plateau로 연장한다. 따라서 다음 셀에서 실제 periodic residual도 확인한다.


In [ ]:
@lru_cache(maxsize=32)
def solve_radial_bvp(a, b, kappa, nominal_radius, rb, nodes, tol, max_nodes):
    phi0 = math.sqrt(-a/b)
    width = math.sqrt(2*kappa/(-a))
    mean_disk = phi0*(2*(nominal_radius/rb)**2 - 1)
    target_integral = mean_disk*rb**2/2
    r = np.linspace(0, rb, nodes)
    S = np.zeros((3,3))
    S[1,1] = -1.0

    def equation(r, y, p):
        phi, derivative, integral = y
        return np.vstack((derivative, (a*phi+b*phi**3-p[0])/kappa, r*phi))

    def boundary(left, right, p):
        return np.array([left[1], right[1], left[2],
                         (right[2]-target_integral)/rb**2])

    failures = []
    for factor in (1.0, 0.9, 1.1):
        z = (factor*nominal_radius-r)/width
        guess_phi = phi0*np.tanh(z)
        guess_derivative = -phi0/width*(1-np.tanh(z)**2)
        guess = np.vstack((guess_phi, guess_derivative,
                           cumulative_trapezoid(r*guess_phi, r, initial=0)))
        mu_guess = kappa*phi0/(nominal_radius*width)
        solution = solve_bvp(equation, boundary, r, guess, p=[mu_guess], S=S,
                             tol=tol, bc_tol=tol, max_nodes=max_nodes)
        dense_r = np.linspace(0, rb, 2001)
        values = solution.sol(dense_r)
        phi = values[0]
        bc_error = float(np.max(np.abs(boundary(solution.y[:,0], solution.y[:,-1], solution.p))))
        rms = float(np.max(solution.rms_residuals))
        droplet_branch = (np.isfinite(values).all() and phi[0] > 0 and phi[-1] < 0
                          and phi[0]-phi[-1] > phi0
                          and np.max(np.diff(phi)) < 1e-3*phi0)
        if solution.success and bc_error < 10*tol and rms < 10*tol and droplet_branch:
            radius = brentq(lambda x: float(solution.sol(x)[0]), 0, rb)
            info = dict(mu=float(solution.p[0]), center=float(phi[0]), exterior=float(phi[-1]),
                        nominal_radius=nominal_radius, crossing_radius=radius,
                        disk_mean=mean_disk, bc_error=bc_error, max_rms_residual=rms,
                        nodes=int(solution.x.size))
            return solution, info
        failures.append(f"status={solution.status}, branch={droplet_branch}, residual={rms:.3g}")
    raise RuntimeError("No accepted droplet BVP branch: " + "; ".join(failures))

coord = (np.arange(n)*dx-length/2+length/2) % length-length/2
X, Y = np.meshgrid(coord, coord)
DISTANCE = np.hypot(X, Y)
THETA = np.arctan2(Y, X)

def local_bvp_field(case):
    sol, info = solve_radial_bvp(case["a"], case["b"], case["kappa"],
        case["radius"], outer_radius, BVP["nodes"], BVP["tol"], BVP["max_nodes"])
    field = sol.sol(np.minimum(DISTANCE, outer_radius).ravel())[0].reshape(n,n)
    return field.copy(), dict(info, square_mean=float(field.mean())), sol

# Editable choice for visualizing the boundary solution.
INITIAL_CASE = next(c for c in CASES if c["name"] == "target") if any(c["name"] == "target" for c in CASES) else CASES[0]
try:
    bvp_field, bvp_info, bvp_solution = local_bvp_field(INITIAL_CASE)
    r = np.linspace(0, outer_radius, 1000)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(r, bvp_solution.sol(r)[0], label="BVP solution")
    phi0 = math.sqrt(-INITIAL_CASE["a"]/INITIAL_CASE["b"])
    axes[0].axhline(phi0, color="gray", ls=":", label="local minima")
    axes[0].axhline(-phi0, color="gray", ls=":")
    axes[0].axvline(INITIAL_CASE["radius"], color="black", ls="--", label="nominal radius")
    axes[0].set(xlabel="r", ylabel="phi", title="Center/exterior Neumann BC + fixed disk mass")
    axes[0].legend()
    im = axes[1].imshow(bvp_field, origin="lower", extent=(0,length,0,length), cmap="RdBu_r")
    axes[1].set(title="Embedded BVP field", xlabel="x", ylabel="y")
    axes[1].grid(False)
    fig.colorbar(im, ax=axes[1], label="phi")
    fig.tight_layout()
    plt.show()
    display_table([bvp_info])
except (RuntimeError, ValueError) as error:
    print("Radial BVP preview failed:", error)
    print("The sweep still attempts all cases; failure is not a stability verdict.")


## 3. Periodic Poisson stationary initialization
The radial boundary problem solves the **local passive continuum** equation. It is a seed for the actual square-grid problem, not an exact nonlocal equilibrium.
At fixed square mean, solve `mu(phi)=constant` with the same discrete Laplacian, bond-gradient activity and Poisson convolution used below. Continue both g and lambda from zero to their requested values.

This nonlinear solve locates a stationary branch; it does not establish stability. It can converge to a saddle. Failure is recorded separately from dynamical breakup. Set `INITIALIZATION="local_bvp"` to study relaxation after switching on the interaction/activity instead.

The nonlocal term is `g*K*phi`, with `Khat=exp(-ell*q)` and lattice wave number q. Coupling g appears exactly once. Removing Khat(0) changes only a spatially constant chemical potential at conserved mass.


In [ ]:
wave = 2*np.pi*np.fft.fftfreq(n, d=dx)
bond = 2*np.sin(wave*dx/2)/dx
QX, QY = np.meshgrid(bond, bond)
Q2 = QX**2 + QY**2

def kernel_multiplier(case):
    multiplier = KernelSpec(kind="poisson", ell=case["ell"]).multiplier(np.sqrt(Q2))
    multiplier[0, 0] = 0.0 if REMOVE_ZERO_MODE else 1.0
    return multiplier

def laplacian_numpy(phi):
    return sum(np.roll(phi, 1, ax) + np.roll(phi, -1, ax) - 2*phi
               for ax in (-2, -1))/dx**2

def grad2_numpy(phi):
    return sum((np.roll(phi, 1, ax)-phi)**2 + (np.roll(phi, -1, ax)-phi)**2
               for ax in (-2, -1))/(2*dx**2)

def chemical_potential(phi, case):
    convolution = np.fft.ifft2(kernel_multiplier(case)*np.fft.fft2(phi)).real
    return (case["a"]*phi + case["b"]*phi**3 - case["kappa"]*laplacian_numpy(phi)
            + case["g"]*convolution + case["lam"]*grad2_numpy(phi))

def residual_report(phi, case):
    mu = chemical_potential(phi, case)
    drift = case["mobility"]*laplacian_numpy(mu)
    return dict(mu=float(mu.mean()), mu_spread=float(np.max(abs(mu-mu.mean()))),
                drift_rms=float(np.sqrt(np.mean(drift**2))), square_mean=float(phi.mean()))

# Reuse deterministic initial states for cases differing only by noise or name.
INITIAL_STATES = {}

def make_initial_state(case):
    key = json.dumps(dict(physics={k:v for k,v in case.items() if k not in ("name", "D")},
                          grid=GRID, bvp=BVP, root=ROOT, initialization=INITIALIZATION,
                          zero_mode=REMOVE_ZERO_MODE), sort_keys=True)
    if key in INITIAL_STATES:
        field, info = INITIAL_STATES[key]
        return field.copy(), dict(info)
    field, radial_info, _ = local_bvp_field(case)
    target_mean = float(field.mean())
    before = residual_report(field, case)
    continuation = []
    if INITIALIZATION == "periodic_stationary":
        previous_mu = float(chemical_potential(field, {**case, "g":0., "lam":0.}).mean())
        fractions = np.linspace(0., 1., ROOT["continuation_steps"]+1)
        if case["g"] == 0 and case["lam"] == 0:
            fractions = [0.]
        for fraction in fractions:
            print(f"  stationary continuation {fraction:.2f}", flush=True)
            current = {**case, "g":fraction*case["g"], "lam":fraction*case["lam"]}
            def equation(z):
                phi = z[:-1].reshape(n, n)
                return np.r_[(chemical_potential(phi, current)-z[-1]).ravel(),
                             n*n*(phi.mean()-target_mean)]
            solved = root(equation, np.r_[field.ravel(), previous_mu], method="krylov",
                          options=dict(fatol=ROOT["tol"], maxiter=ROOT["maxiter"]))
            error = float(np.max(abs(equation(solved.x))))
            if not solved.success or not np.isfinite(error) or error > 5*ROOT["tol"]:
                raise RuntimeError(f"Periodic initialization failed at continuation={fraction:.3f}: "
                                   f"residual={error:.3g}; {solved.message}")
            field = solved.x[:-1].reshape(n, n)
            previous_mu = float(solved.x[-1])
            morphology = cluster_observables(field[None], dx, threshold=0., modes=2)
            phi0 = np.sqrt(-case["a"]/case["b"])
            if (field[n//2,n//2] <= 0 or field[0,0] >= 0
                or field.max()-field.min() < phi0
                or morphology["components"][0] != 1
                or morphology["coverage"][0] < 1):
                raise RuntimeError(f"Continuation={fraction:.3f} left the resolved droplet branch.")
            continuation.append(dict(fraction=float(fraction), residual=error,
                                     iterations=int(getattr(solved, "nit", -1))))
    after = residual_report(field, case)
    if abs(field.mean()-target_mean) > RULES["mass_tolerance"]:
        raise RuntimeError("Stationary initialization did not preserve the target square mean.")
    info = dict(radial=radial_info, before=before, after=after,
                continuation=continuation, initialization=INITIALIZATION)
    INITIAL_STATES[key] = (field.copy(), info)
    return field, info


## 4. Equal-mass shape perturbation
Deform the interface by a small cos(m theta) displacement, with a smooth radial envelope that vanishes before the periodic boundary. A scalar radial correction restores the reference mass; the bulk field is not uniformly shifted.
The reference and perturbed member receive identical conserved noise at each step. At D=0 this is a deterministic stability probe; one noisy pair cannot establish a survival probability.


In [ ]:
def shape_perturbation(reference):
    obs = cluster_observables(reference[None], dx, threshold=0., modes=2)
    radius = float(obs["mean_interface_radius"][0])
    if not np.isfinite(radius) or not 3*dx < radius < length/3:
        raise ValueError("Initial droplet is not resolved or is too close to the box boundary.")
    cutoff = (radius+length/2)/2
    envelope = np.zeros_like(DISTANCE)
    mask = DISTANCE < cutoff
    r = DISTANCE[mask]
    mode = int(PERTURB["mode"])
    envelope[mask] = (r/radius)**mode * ((1-(r/cutoff)**2)/(1-(radius/cutoff)**2))**2
    angular = PERTURB["relative_amplitude"]*radius*np.cos(mode*THETA)
    def deform(shift):
        source_radius = DISTANCE-(shift+angular)*envelope
        if np.min(source_radius) < -1e-12:
            raise ValueError("Perturbation folds radial coordinates; reduce its amplitude.")
        scale = np.divide(source_radius, DISTANCE, out=np.ones_like(DISTANCE),
                          where=DISTANCE>0)
        coordinates = [(length/2+scale*Y)/dx, (length/2+scale*X)/dx]
        return map_coordinates(reference, coordinates, order=3, mode="grid-wrap")
    target = reference.mean()
    def mass_error(shift):
        return float(deform(shift).mean()-target)
    bracket = PERTURB["relative_amplitude"]*radius
    for _ in range(6):
        if mass_error(-bracket)*mass_error(bracket) <= 0:
            break
        bracket *= 1.5
    else:
        raise RuntimeError("Could not bracket the mass-preserving shape correction.")
    shift = brentq(mass_error, -bracket, bracket, xtol=1e-12)
    perturbed = deform(shift)
    error = float(perturbed.mean()-target)
    if abs(error) > RULES["mass_tolerance"]:
        raise RuntimeError("Shape perturbation mass mismatch.")
    return perturbed, dict(radial_correction=float(shift), mass_error=error,
                           reference_radius=radius)

# Initialization preview: failure here is not labelled a dynamical instability.
try:
    preview, preview_info = make_initial_state(INITIAL_CASE)
    preview_perturbed, perturb_info = shape_perturbation(preview)
    display_table([dict(stage="embedded local BVP", **preview_info["before"]),
                   dict(stage="chosen initialization", **preview_info["after"])])
    display_table([perturb_info])
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, field, title in zip(axes, [preview, preview_perturbed,
                                      chemical_potential(preview, INITIAL_CASE)],
                                ["Reference", "Equal-mass perturbed", "Initial chemical potential"]):
        im = ax.imshow(field, origin="lower", extent=[0,length,0,length], cmap="coolwarm")
        ax.set(title=title, xlabel="x", ylabel="y")
        fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()
except (RuntimeError, ValueError) as error:
    print("Target initialization unavailable:", error)
    print("The sweep below still attempts the other cases.")


## 5. Euler–Maruyama evolution (editable solver cell)
The two fields are advanced together in float64 on CUDA when available. Every step recomputes the Fourier transform of the current field and the nonlinear chemical potential. Only the fixed q grid and kernel multiplier are cached.

Noise is a real-space unit Gaussian transformed to Fourier space and multiplied by q*sqrt(2*M*D*dt/dx²). Its zero mode vanishes. **D=0 is exactly deterministic.** No entropy calculation is needed here.

Frames are stored as float32, while mass diagnostics are recorded from the float64 state. This is a morphology study, not a measurement of steady entropy production. A fixed observation horizon is not proof of asymptotic stability.


In [ ]:
@torch.no_grad()
def evolve_pair(reference, perturbed, case):
    dtype = torch.float64
    phi = torch.as_tensor(np.stack([reference, perturbed]), dtype=dtype, device=DEVICE)
    q2 = torch.as_tensor(Q2, dtype=dtype, device=DEVICE)
    kernel = torch.as_tensor(kernel_multiplier(case), dtype=dtype, device=DEVICE)
    linear = case["a"] + case["kappa"]*q2 + case["g"]*kernel
    noise_scale = torch.sqrt(q2*(2*case["mobility"]*case["D"]*RUN["dt"]/dV))
    rng = torch.Generator(device=DEVICE).manual_seed(RUN["seed"])
    initial_mean = phi.mean(dim=(-2,-1)).clone()
    # Orthonormal FFT in 2D: zero coefficient = mean*n.
    zero_coefficient = initial_mean*n
    saved_fields, saved_times, saved_means = [], [], []
    def record(step):
        saved_fields.append(phi.to(dtype=torch.float32).cpu().numpy().copy())
        saved_times.append(step*RUN["dt"])
        saved_means.append(phi.mean(dim=(-2,-1)).cpu().numpy().copy())
    record(0)
    for step in tqdm(range(1, steps+1), desc=case["name"], leave=False):
        phi_hat = torch.fft.fft2(phi, norm="ortho")
        grad2 = sum((torch.roll(phi, 1, ax)-phi).square()
                    + (torch.roll(phi, -1, ax)-phi).square()
                    for ax in (-2,-1))/(2*dx**2)
        nonlinear = case["b"]*phi.pow(3) + case["lam"]*grad2
        mu_hat = linear*phi_hat + torch.fft.fft2(nonlinear, norm="ortho")
        next_hat = phi_hat - RUN["dt"]*case["mobility"]*q2*mu_hat
        if case["D"] > 0:
            # Same real Gaussian field for both members, independent at each step.
            noise = torch.randn((n,n), generator=rng, device=DEVICE, dtype=dtype)
            next_hat = next_hat + noise_scale*torch.fft.fft2(noise, norm="ortho")
        next_hat[:,0,0] = zero_coefficient
        phi = torch.fft.ifft2(next_hat, norm="ortho").real
        if step % 100 == 0 or step == steps:
            if not bool(torch.isfinite(phi).all()):
                raise FloatingPointError(f"Nonfinite Euler state at t={step*RUN['dt']}; decrease dt.")
            if float((phi.mean(dim=(-2,-1))-initial_mean).abs().max()) > RULES["mass_tolerance"]:
                raise FloatingPointError("Mass conservation tolerance exceeded.")
        if step % stride == 0 or step == steps:
            record(step)
    return dict(fields=np.stack(saved_fields), time=np.array(saved_times),
                mass_mean=np.stack(saved_means))

def cache_description(case, reference, perturbed):
    # Change NUMERICS_VERSION when editing equations or noise conventions.
    initial_hash = hashlib.sha256(np.stack([reference,perturbed]).tobytes()).hexdigest()
    description = dict(version=NUMERICS_VERSION, case=case, grid=GRID, run=RUN,
                       initialization=INITIALIZATION, bvp=BVP, root=ROOT, perturb=PERTURB,
                       remove_zero_mode=REMOVE_ZERO_MODE, initial_sha256=initial_hash,
                       device=str(DEVICE), torch_version=torch.__version__)
    serialized = json.dumps(description, sort_keys=True)
    return description, hashlib.sha256(serialized.encode()).hexdigest()[:20]

RESULTS, FAILURES = {}, []
for case in CASES:
    stage = "initialization"
    try:
        print(f"Preparing {case['name']}: g={case['g']}, ell={case['ell']}, "
              f"lambda={case['lam']}, D={case['D']}, nominal R={case['radius']}", flush=True)
        reference, init_info = make_initial_state(case)
        perturbed, perturb_info = shape_perturbation(reference)
        description, signature = cache_description(case, reference, perturbed)
        path = OUTPUT_ROOT / f"pair_{signature}.npz"
        stage = "simulation/cache"
        if path.exists():
            with np.load(path, allow_pickle=False) as saved:
                trajectory = {key:saved[key].copy() for key in ("fields","time","mass_mean")}
            print("Loaded", path.name)
        else:
            trajectory = evolve_pair(reference, perturbed, case)
            # Atomic replacement: interruption cannot leave a partial final archive.
            temporary = path.with_suffix(".tmp")
            try:
                with temporary.open("wb") as handle:
                    np.savez_compressed(handle, **trajectory,
                        metadata=json.dumps(dict(description=description, initialization=init_info,
                                                 perturbation=perturb_info)))
                temporary.replace(path)
            except OSError as error:
                print(f"Could not cache {path}: {error}. This trajectory remains in RESULTS.")
                if temporary.exists():
                    temporary.unlink()
        RESULTS[case["name"]] = dict(case=case, initial=reference, perturbed_initial=perturbed,
                                     initialization=init_info, perturbation=perturb_info,
                                     trajectory=trajectory, path=path)
    except (RuntimeError, ValueError, FloatingPointError) as error:
        FAILURES.append(dict(name=case["name"], stage=stage, message=str(error)))
        print(f"{stage} FAILED for {case['name']}: {error}")
if FAILURES:
    display_table(FAILURES)
print(f"Completed {len(RESULTS)}/{len(CASES)} cases; cached sparse frames in {OUTPUT_ROOT}")


## 6. Morphology and finite-time classification
All contours use phi=0, between the two phases of the local double well. Centres and components respect periodic boundaries. Circularity is 4*pi*area/perimeter² of the interpolated radial contour; no valid closed contour means NaN, not a circular droplet.

The paired shape diagnostic subtracts the reference mode after centring each droplet and normalizing by its own instantaneous radius. Its fitted log-growth slope is descriptive; adjacent times are correlated, so no independent-sample confidence interval is attached.

"Finite-time circular survival" requires the last quarter of the record to remain within the chosen radius tolerance, with a sufficiently circular dominant component. Growth/shrinkage, deformation, fragmentation, and missing interfaces are reported separately. Inspect both members and the movie, especially at D>0.


In [ ]:
def morphology(fields, case):
    obs = cluster_observables(fields, dx, threshold=0., angles=128,
                              modes=max(6, int(PERTURB["mode"])), min_sites=8)
    boundary = obs["interface_radius"]
    next_boundary = np.roll(boundary, -1, axis=1)
    angle = 2*np.pi/boundary.shape[1]
    perimeter = np.sqrt(np.maximum(0., boundary**2+next_boundary**2
                        -2*boundary*next_boundary*np.cos(angle))).sum(axis=1)
    area = 0.5*np.sin(angle)*(boundary*next_boundary).sum(axis=1)
    circularity = np.divide(4*np.pi*area, perimeter**2,
                           out=np.full_like(area,np.nan), where=perimeter>0)
    valid = (obs["coverage"] == 1) & np.isfinite(obs["shape"][:,0])
    obs["circularity"] = np.where(valid, circularity, np.nan)
    inside, outside = [], []
    width = np.sqrt(2*case["kappa"]/(-case["a"]))
    for phi, centre, radius in zip(fields, obs["centre"], obs["radius"]):
        x = (np.arange(n)*dx-centre[0]+length/2)%length-length/2
        y = (np.arange(n)*dx-centre[1]+length/2)%length-length/2
        xx, yy = np.meshgrid(x,y)
        rr = np.hypot(xx,yy)
        core, exterior = rr < .5*radius, rr > radius+4*width
        inside.append(float(phi[core].mean()) if core.any() and radius>2*dx else np.nan)
        outside.append(float(phi[exterior].mean()) if exterior.any() else np.nan)
    obs["bulk_inside"], obs["bulk_outside"] = np.array(inside), np.array(outside)
    return obs

def survival_label(obs):
    tail = slice(max(0,len(obs["radius"])*3//4),None)
    radius_ratio = obs["radius"][tail]/max(obs["radius"][0],dx)
    circ = obs["circularity"][tail]
    if np.median(radius_ratio) < .2:
        return "dissolved"
    if np.any(obs["components"][tail] > 1):
        return "fragmented"
    if np.any(~np.isfinite(circ)):
        return "unresolved/noncircular interface"
    if np.any(obs["largest_fraction"][tail] < RULES["largest_fraction_min"]):
        return "secondary positive-phase regions"
    if np.any(circ < RULES["circularity_min"]):
        return "deformed"
    if np.any(radius_ratio < 1-RULES["radius_tolerance"]):
        return "radial shrinkage"
    if np.any(radius_ratio > 1+RULES["radius_tolerance"]):
        return "radial growth"
    return "finite-time circular survival"

SUMMARY_ROWS = []
for name, result in RESULTS.items():
    trajectory, case = result["trajectory"], result["case"]
    time_axis = trajectory["time"]
    obs = [morphology(trajectory["fields"][:,j], case) for j in (0,1)]
    a, b = obs
    mode = int(PERTURB["mode"])
    ar, br = a["mean_interface_radius"], b["mean_interface_radius"]
    normalized_a = np.divide(a["shape"][:,mode], ar,
                            out=np.full(ar.shape,complex(np.nan,np.nan)), where=ar>0)
    normalized_b = np.divide(b["shape"][:,mode], br,
                            out=np.full(br.shape,complex(np.nan,np.nan)), where=br>0)
    amplitude = 2*abs(normalized_b-normalized_a)
    gain = amplitude/amplitude[0] if amplitude[0]>1e-12 else np.full_like(amplitude,np.nan)
    fit_valid = ((a["components"]==1)&(b["components"]==1)&np.isfinite(amplitude)
                 &(amplitude>1e-12)&(amplitude<RULES["shape_linear_limit"])
                 &(np.minimum(ar,br)>3*dx)&(time_axis<=RULES["fit_until"]))
    first_bad = np.flatnonzero(~fit_valid)
    fit_count = int(first_bad[0]) if first_bad.size else len(time_axis)
    slope = float(np.polyfit(time_axis[:fit_count],np.log(amplitude[:fit_count]),1)[0]) if fit_count>=6 else np.nan
    labels = [survival_label(o) for o in obs]
    result["diagnostics"] = dict(reference=a, perturbed=b, gain=gain,
                                  relative_amplitude=amplitude, log_growth_rate=slope,
                                  fit_points=fit_count, labels=labels)
    mass_drift = np.max(abs(trajectory["mass_mean"]-trajectory["mass_mean"][0]))
    SUMMARY_ROWS.append(dict(name=name, g=case["g"], ell=case["ell"], lam=case["lam"],
        D=case["D"], nominal_R=case["radius"], initial_R=float(a["radius"][0]),
        mean=float(trajectory["mass_mean"][0,0]),
        initial_mu_spread=result["initialization"]["after"]["mu_spread"],
        final_radius_ratio=float(a["radius"][-1]/a["radius"][0]),
        final_circularity=float(a["circularity"][-1]), max_mass_drift=float(mass_drift),
        paired_mode_log_rate=slope, fit_points=fit_count,
        reference_status=labels[0], perturbed_status=labels[1]))
SUMMARY = SUMMARY_ROWS
display_table(SUMMARY)
if FAILURES:
    display_table(FAILURES)
# Optional persistent summary after inspecting the results:
# (OUTPUT_ROOT / "summary.json").write_text(json.dumps(SUMMARY, indent=2), encoding="utf-8")


## 7. Direct matplotlib cells: snapshots and time histories
Edit `PLOT_NAMES`, the selected member, axis limits and labels here. Only saved frames are used: these cells do not rerun dynamics.
The defaults compare the controls and target. For interaction strength alone use target/weaker g/stronger g; for length alone use target/shorter ell=2/shorter ell=4.
Changing nominal radius changes conserved mass, so that comparison is a droplet-size experiment rather than a fixed-mass comparison.


In [ ]:
PLOT_NAMES = [name for name in ("local passive", "local active", "Poisson passive", "target")
              if name in RESULTS]
# PLOT_NAMES = [name for name in ("weaker g", "target", "stronger g") if name in RESULTS]
# PLOT_NAMES = [name for name in ("shorter ell=2", "shorter ell=4", "target") if name in RESULTS]
# PLOT_NAMES = list(RESULTS)
if not PLOT_NAMES and RESULTS:
    PLOT_NAMES = [next(iter(RESULTS))]
if PLOT_NAMES:
    bound = max(float(abs(RESULTS[name]["trajectory"]["fields"]).max()) for name in PLOT_NAMES)
    fig, axes = plt.subplots(len(PLOT_NAMES), 3, figsize=(12, 3.4*len(PLOT_NAMES)), squeeze=False)
    for row, name in enumerate(PLOT_NAMES):
        result = RESULTS[name]
        fields = result["trajectory"]["fields"]
        panels = [fields[0,0], fields[-1,0], fields[-1,1]]
        titles = ["Initial reference", "Final reference", "Final perturbed"]
        for ax, field, title in zip(axes[row], panels, titles):
            im = ax.imshow(field, origin="lower", extent=(0,length,0,length),
                           cmap="RdBu_r", vmin=-bound, vmax=bound)
            if field.min() < 0 < field.max():
                ax.contour(np.arange(n)*dx, np.arange(n)*dx, field,
                           levels=[0], colors="black", linewidths=.6)
            ax.set(title=f"{name}: {title}", xlabel="x", ylabel="y")
            ax.grid(False)
    fig.colorbar(im, ax=axes.ravel().tolist(), label="phi", shrink=.8)
    plt.show()
else:
    print("No completed trajectories; inspect initialization failures above.")


In [ ]:
MEMBER = "reference"  # or "perturbed"
if PLOT_NAMES:
    fig, axes = plt.subplots(2, 3, figsize=(16,8), constrained_layout=True)
    colors = plt.cm.tab20(np.linspace(0,1,len(PLOT_NAMES)))
    for name, color in zip(PLOT_NAMES, colors):
        result = RESULTS[name]
        t = result["trajectory"]["time"]
        obs = result["diagnostics"][MEMBER]
        axes[0,0].plot(t, obs["radius"]/obs["radius"][0], color=color, label=name)
        axes[0,1].plot(t, obs["circularity"], color=color)
        axes[0,2].plot(t, obs["components"], color=color)
        axes[1,0].semilogy(t, result["diagnostics"]["gain"], color=color)
        axes[1,1].plot(t, obs["bulk_inside"], color=color)
        axes[1,1].plot(t, obs["bulk_outside"], color=color, ls="--")
        member = 0 if MEMBER=="reference" else 1
        means = result["trajectory"]["mass_mean"][:,member]
        axes[1,2].plot(t, means-means[0], color=color)
    axes[0,0].axhspan(1-RULES["radius_tolerance"],1+RULES["radius_tolerance"],
                     color="gray",alpha=.1)
    axes[0,1].axhline(RULES["circularity_min"], color="gray", ls=":")
    axes[1,0].axhline(1, color="gray", ls=":")
    titles = ["Area radius / initial radius", "Contour circularity", "Resolved components",
              f"Paired m={PERTURB['mode']} relative-mode gain",
              "Core (solid) / exterior (dashed) bulk values", "Conserved mean drift (float64)"]
    for ax, title in zip(axes.ravel(), titles):
        ax.set(xlabel="time", title=title)
    axes[0,0].legend(fontsize=8)
    plt.show()


## 8. Inspect one case: radial profile, contour and profile movie
A flat mean radius alone is insufficient: inspect bulk relaxation and the shape perturbation too.
Edit `INSPECT_NAME` and `FRAME_INDICES`. The movie uses the same fixed phi scale for reference and perturbed fields; the difference panel has its own fixed scale.
Saved times include t=0, with no hidden burn-in.


In [ ]:
INSPECT_NAME = "target" if "target" in RESULTS else next(iter(RESULTS), None)
if INSPECT_NAME is not None:
    result = RESULTS[INSPECT_NAME]
    t = result["trajectory"]["time"]
    ref = result["diagnostics"]["reference"]
    per = result["diagnostics"]["perturbed"]
    FRAME_INDICES = sorted(set([0, len(t)//4, len(t)//2, len(t)-1]))
    fig, axes = plt.subplots(1,3,figsize=(15,4),constrained_layout=True)
    angle = np.linspace(0,2*np.pi,ref["interface_radius"].shape[1],endpoint=False)
    for index in FRAME_INDICES:
        axes[0].plot(ref["radial_coordinate"],ref["radial_profile"][index],
                     label=f"t={t[index]:g}")
        axes[1].plot(angle, ref["interface_radius"][index], label=f"t={t[index]:g}")
        axes[2].plot(angle, per["interface_radius"][index]-ref["interface_radius"][index],
                     label=f"t={t[index]:g}")
    phi0 = np.sqrt(-result["case"]["a"]/result["case"]["b"])
    for plateau in (-phi0,phi0):
        axes[0].axhline(plateau,color="gray",ls=":",lw=.8)
    axes[0].set(xlabel="distance from tracked centre",ylabel="azimuthal mean phi",title=INSPECT_NAME)
    axes[1].set(xlabel="angle",ylabel="interface radius",title="Reference contour")
    axes[2].set(xlabel="angle",ylabel="perturbed - reference radius",title="Paired shape difference")
    axes[0].legend()
    plt.show()


In [ ]:
MOVIE_FPS = 12
MAX_MOVIE_FRAMES = 161
if INSPECT_NAME is not None:
    result = RESULTS[INSPECT_NAME]
    fields, movie_times = result["trajectory"]["fields"], result["trajectory"]["time"]
    movie_indices = np.unique(np.linspace(0,len(fields)-1,min(MAX_MOVIE_FRAMES,len(fields))).astype(int))
    differences = fields[:,1]-fields[:,0]
    bound = float(np.max(abs(fields)))
    diff_bound = max(float(np.max(abs(differences))),1e-8)
    fig, axes = plt.subplots(1,3,figsize=(13,4),constrained_layout=True)
    movie_images = []
    for ax, field, limit, title in zip(axes, [fields[0,0],fields[0,1],differences[0]],
                                     [bound,bound,diff_bound],
                                     ["Reference","Same-mass perturbed","Perturbed - reference"]):
        image = ax.imshow(field,origin="lower",extent=(0,length,0,length),
                          cmap="RdBu_r",vmin=-limit,vmax=limit)
        ax.set(title=title,xlabel="x",ylabel="y")
        ax.grid(False)
        fig.colorbar(image,ax=ax,shrink=.75)
        movie_images.append(image)
    movie_title = fig.suptitle("")
    def update_movie(frame):
        index = int(movie_indices[frame])
        for image, field in zip(movie_images,[fields[index,0],fields[index,1],differences[index]]):
            image.set_data(field)
        obs = result["diagnostics"]["reference"]
        movie_title.set_text(f"{INSPECT_NAME} | t={movie_times[index]:.2f} | "
                             f"R={obs['radius'][index]:.2f}, C={obs['circularity'][index]:.3f}")
        return [*movie_images,movie_title]
    profile_movie = animation.FuncAnimation(fig,update_movie,frames=len(movie_indices),
                                            interval=1000/MOVIE_FPS,blit=False)
    plt.close(fig)
    display(HTML(profile_movie.to_jshtml(fps=MOVIE_FPS)))
    # Optional export (ffmpeg must be installed):
    # profile_movie.save(OUTPUT_ROOT / "profile_pair.mp4", fps=MOVIE_FPS, dpi=120)


## Reading the result / next parameter change

- **Reference survives, paired mode decays:** evidence of finite-time stability to the selected shape mode at this mass and resolution.
- **Reference survives, paired mode grows:** stationary or slowly evolving circular profile can still be unstable to shape perturbations.
- **Radius changes without much deformation:** radial mass redistribution; a circular-looking droplet need not be stationary.
- **Initialization fails:** change continuation steps/root tolerance, nominal radius or parameters, or explicitly select the local-BVP quench. Do not interpret this alone as droplet instability.
- **D>0:** the deterministic stationary state is only an initial condition, not a draw from the stationary noisy distribution. Use several RUN seeds for survival statistics.
- **Numerical convergence:** before drawing physical conclusions, compare smaller dt, finer dx at fixed physical L/R/ell and a larger physical box. Reducing dx usually requires a much smaller Euler dt. The notebook does not automatically run these studies.

The local BVP imposes no-flux/regularity conditions on a disk. Subsequent dynamics use a periodic square, with **no externally pinned circular wall**. The periodic refinement explicitly removes the discretization/geometry residual where it converges. For a common starting profile across interaction conditions, choose local_bvp and keep a/b/kappa/radius fixed.

Edit numerical equations above directly. Update `NUMERICS_VERSION` after changing them so old trajectory caches are not reused. Plot edits do not require simulation.
